# Charge-blocked vs dense two-site splits in the walker conversion

`mps_cpmc_new.py` turns every walker spin channel ($L\times N$ orthonormal $Q$) into a $d=2$ MPS by applying Fishman–White Givens gates to an occupation product state (`mps_cpmc_walkthrough.ipynb`). After each gate the two-site tensor $T_{a\,n_p\,n_{p+1}\,b}$ (shape $D_l\times2\times2\times D_r$) is reshaped to the matrix

\begin{equation*}
M_{(a n_p),(n_{p+1} b)} = T_{a\,n_p\,n_{p+1}\,b},\qquad M\in\mathbb R^{2D_l\times 2D_r},
\end{equation*}

and split, $M\approx A B$, into a left isometry $A$ and the new orthogonality centre $B$. `m.split_pair` does this charge block by charge block. This notebook measures, on the real matrices of real conversions, how much faster that is than the naive alternative (one SVD of the whole $M$), and explains the result.

**Why $M$ is block diagonal.** Label a left bond index $a$ by its charge $q_l[a]$, the particle number on sites $0,\dots,p-1$, and a right bond index $b$ by $q_r[b]$, the particle number on sites $0,\dots,p+1$. Every gate conserves particle number, so $T_{a\,n_p\,n_{p+1}\,b}\neq0$ only if $q_l[a]+n_p+n_{p+1} = q_r[b]$, i.e.

\begin{equation*}
\underbrace{q_l[a]+n_p}_{\text{row charge}} \;=\; \underbrace{q_r[b]-n_{p+1}}_{\text{column charge}} .
\end{equation*}

Sorting rows and columns by charge makes $M = \bigoplus_q M_q$, one block per value $q$ of the particle number on sites $0,\dots,p$, which is the charge of the new middle bond. The SVD of $M$ is the direct sum of the blocks' SVDs, and every singular vector carries a definite charge. `m.sector_plan(ql, qr, kept)` lists the blocks as (row indices, column indices, rank).

**What `m.split_pair` does.** It factorises each block separately, keeps the frozen number `kept[s]` of vectors in block $s$ (all of them for an exact split, `kept=None`), scatters the blocks back into full row and column order (`m._assemble`), and gives the middle bond the labels `plan.middle_charges`. The next centre move, the charge-blocked contraction with the trial and the static shapes under `jax.jit` all need those labels. Which dense kernel it runs per block is an implementation detail, and it changed while this notebook was written: from a QR per block plus eigh of $R_qR_q^T$ when truncating, to a QR per untruncated block, one eigh of the smaller Gram matrix ($M_qM_q^T$ or $M_q^TM_q$) per truncated block, and closed forms for blocks with one row or column. This notebook only ever *calls* `m.split_pair`, so it measures whatever the imported module does, next to fixed reference implementations.

**Cost model.** A dense SVD of an $m\times n$ matrix costs $\sim c\,mn\min(m,n)$ flops. The blocked split costs $\sum_q c\,m_qn_q\min(m_q,n_q)$, which for $S$ equal blocks is $1/S^2$ of the dense cost. But it replaces one LAPACK call by at least one per block (a QR alone is two, `geqrf` and `orgqr`), plus gathers and a scatter. When the matrices are small, per-call overhead rather than flops sets the time, and that can hand the win to the dense SVD. Which regime the conversion is in is an empirical question, answered below.

In [2]:
import os
import platform
import time

import numpy as np
import scipy.linalg
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import mps_cpmc_new as m                     # also enables jax x64

np.set_printoptions(precision=4, suppress=True, linewidth=110)
print("jax", jax.__version__, "| numpy", np.__version__, "|", platform.machine(), platform.mac_ver()[0],
      "|", os.cpu_count(), "cores | x64:", jax.config.jax_enable_x64)
print("thread env:", {k: os.environ.get(k) for k in ("OMP_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "XLA_FLAGS")})

jax 0.11.1 | numpy 2.5.3 | arm64 26.5.1 | 10 cores | x64: True
thread env: {'OMP_NUM_THREADS': None, 'VECLIB_MAXIMUM_THREADS': None, 'XLA_FLAGS': None}


## Walkers

The reference determinant is the RHF ground state of the open $L$-site chain at half filling. Walkers are propagated from it for 100 CPMC steps, $Q\to e^{-\Delta\tau K/2}\,e^{\gamma\,\mathrm{diag}(s)}\,e^{-\Delta\tau K/2}\,Q$ with random fields $s_i=\pm1$, $\cosh\gamma = e^{\Delta\tau U/2}$ ($\Delta\tau=0.01$, $U=4$), and re-orthonormalised by QR after every step. One spin channel is enough: the other has the same plan at half filling.

In [3]:
def cpmc_walkers(L, n_walkers, steps=100, dt=0.01, U=4.0, seed=0):
    # reference RHF determinant of the open chain, and walkers propagated from it
    h = m.hopping_matrix(L, 1.0)
    C_ref = np.linalg.eigh(h)[1][:, :L // 2]
    half = scipy.linalg.expm(-0.5 * dt * h)
    gamma = np.arccosh(np.exp(dt * U / 2))
    rng = np.random.default_rng(seed)
    walkers = []
    for _ in range(n_walkers):
        Q = C_ref.copy()
        for _ in range(steps):
            s = rng.choice([-1.0, 1.0], size=L)
            Q = np.linalg.qr(half @ (np.exp(gamma * s)[:, None] * (half @ Q)))[0]
        walkers.append(Q)
    return C_ref, np.stack(walkers)


C_ref, W = cpmc_walkers(32, 8)
overlaps = np.abs([np.linalg.det(C_ref.T @ Q) for Q in W])
print("|<C_ref|Q>| for 8 walkers at L=32:", overlaps)
np.allclose(np.einsum("wia,wib->wab", W, W), np.eye(W.shape[2])) and overlaps.max() < 0.99

|<C_ref|Q>| for 8 walkers at L=32: [0.2254 0.2055 0.1173 0.1605 0.1488 0.2134 0.1181 0.1554]


np.True_

## Recording the splits of a conversion

`m.channel_mps` looks up `split_pair` in the module namespace at call time, so temporarily replacing `m.split_pair` with a recorder captures the input of every split, in order, and leaves the conversion unchanged.

To record a batch of walkers at once, the same recorder runs under `jax.vmap` without `jit`. That still executes op by op, but every op acts on the whole batch, so recording 200 walkers costs about as much as recording one. Inside `vmap` the recorded $T$ are batched tracers; they are returned as `vmap` outputs. The plan is frozen, so every walker has the same shapes at the same split index.

The check compares the batch against a plain eager recording. For the reference determinant the two agree to the last bit. For a propagated walker the labels agree but individual $T$ entries drift apart, up to $O(1)$. That is not the recorder: the adaptive plan's `m.channel_angles` picks an eigenvector at the edge of a nearly degenerate eigenspace of the walker's block (several eigenvalues within $10^{-10}$ to $10^{-6}$ of 0 or 1 on these walkers), so the $10^{-16}$ differences between a batched and a single LAPACK call select different, equally good gate angles. On the reference the plan's block sizes are chosen so that the extreme eigenvalue is isolated. Either way the recorded matrices are those of a real conversion, which is all the timing needs.

In [4]:
class Split:
    # one split of a conversion: the batch of two-site tensors and its static labels
    def __init__(self, T, ql, qr, kept):
        self.T, self.ql, self.qr, self.kept = T, ql, qr, kept
        self.plan = m.sector_plan(ql, qr, kept)
        self.k = len(self.plan.middle_charges)                  # vectors kept in total
        self.shape = (2 * T.shape[-4], 2 * T.shape[-1])         # shape of M


def record_splits(Q, plan, bond_plan):
    # run m.channel_mps on one walker; return the input (T, ql, qr, kept) of every split
    recorded = []
    original = m.split_pair

    def recorder(T, ql, qr, kept=None):
        recorded.append((T, ql, qr, kept))
        return original(T, ql, qr, kept)

    m.split_pair = recorder
    try:
        m.channel_mps(jnp.asarray(Q), plan, bond_plan)
    finally:
        m.split_pair = original
    return recorded


def record_batch(walkers, plan, bond_plan):
    # record_splits for a stack of walkers at once
    labels = []

    def tensors(Q):
        recorded = record_splits(Q, plan, bond_plan)
        labels[:] = [(ql, qr, kept) for _, ql, qr, kept in recorded]
        return [T for T, *_ in recorded]

    batches = jax.vmap(tensors)(jnp.asarray(walkers))
    return [Split(np.asarray(T), *lab) for T, lab in zip(batches, labels)]


plan = m.make_orbital_plan(C_ref, "adaptive")
bond_plan = m.plan_bonds(C_ref, plan, 8)
batch = record_batch(np.concatenate([C_ref[None], W]), plan, bond_plan)    # walker 0: the reference
print(len(batch), "splits per conversion at L=32, chi_w=8; T of split 100 has shape", batch[100].T.shape)


def batch_vs_eager(i, Q):
    # labels identical?  and the largest |T_batch - T_eager| of every split, for walker i of the batch
    eager = record_splits(Q, plan, bond_plan)
    same = all(s.kept == kept and np.array_equal(s.ql, ql) and np.array_equal(s.qr, qr)
               for s, (_, ql, qr, kept) in zip(batch, eager))
    return same, np.array([np.abs(s.T[i] - np.asarray(T)).max() for s, (T, *_) in zip(batch, eager)])


same_ref, diff_ref = batch_vs_eager(0, C_ref)
same_w, diff_w = batch_vs_eager(1, W[0])
print(f"reference:          max |T_batch - T_eager| = {diff_ref.max():.1e}")
print(f"propagated walker:  median {np.median(diff_w):.1e}, max {diff_w.max():.1e}")
same_ref and same_w and diff_ref.max() < 1e-12

184 splits per conversion at L=32, chi_w=8; T of split 100 has shape (9, 8, 2, 2, 8)


KeyboardInterrupt: 

## The configurations

L=32 and L=48 with the adaptive orbital plan and walker bond dimensions $\chi_w = 4, 8, 16$, as in production. L=32 also gets $\chi_w=32, 64$, to see where the trend goes for larger matrices. The exact conversion (`bond_plan=None`, no truncation) is done at L=20, where full rank is still affordable. Each configuration records the splits of the conversion of 200 walkers.

In [ ]:
GRID = [(20, None),
        (32, 4), (32, 8), (32, 16), (32, 32), (32, 64),
        (48, 4), (48, 8), (48, 16)]                              # (L, chi_w); chi_w=None is the exact conversion
N_BATCH = 200                        # walkers per jit(vmap) call, as in production

configs = {}
for L, chi in GRID:
    t0 = time.perf_counter()
    C_ref, W = cpmc_walkers(L, N_BATCH, seed=L)
    plan = m.make_orbital_plan(C_ref, "adaptive")
    bond_plan = None if chi is None else m.plan_bonds(C_ref, plan, chi)
    configs[L, chi] = dict(C_ref=C_ref, plan=plan, bond_plan=bond_plan,
                           splits=record_batch(W, plan, bond_plan))
    size = sum(s.T.nbytes for s in configs[L, chi]["splits"]) / 1e6
    print(f"L={L:2d} chi_w={str(chi):>4}: {len(configs[L, chi]['splits']):3d} splits, "
          f"{size:6.0f} MB of recorded T, {time.perf_counter() - t0:5.1f} s")

## What the matrices look like

For each configuration: the number of splits per conversion; the shape of $M$ (largest, and the median number of entries); blocks per split; the largest block; and two ratios that set the stakes. *entries in blocks* is $\sum_q m_qn_q / mn$, summed over the conversion: the fraction of $M$ that can be nonzero. *flop ratio* is $\sum mn\min(m,n) / \sum_q m_qn_q\min(m_q,n_q)$ over the conversion: the speedup blocking would give if time were proportional to SVD flops. The check confirms, on every recorded matrix of every walker, that $M$ is exactly zero outside its charge blocks.

In [ ]:
def dense_flops(s):
    rows, cols = s.shape
    return rows * cols * min(rows, cols)


def block_flops(s):
    return sum(len(r) * len(c) * min(len(r), len(c)) for r, c, _ in s.plan.sectors)


def outside_blocks(s):
    # largest |M| outside the charge blocks, over all walkers of the batch
    M = s.T.reshape(len(s.T), *s.shape)
    mask = np.zeros(s.shape, bool)
    for r, c, _ in m.sector_plan(s.ql, s.qr).sectors:
        mask[np.ix_(r, c)] = True
    return np.abs(M[:, ~mask]).max(initial=0.0)


print(" L  chi_w  splits  largest M  median entries  blocks/split (max)  largest block  "
      "entries in blocks  flop ratio")
zero_outside = []
for (L, chi), cfg in configs.items():
    S = cfg["splits"]
    largest = max(S, key=lambda s: s.shape[0] * s.shape[1]).shape
    blocks = [len(s.plan.sectors) for s in S]
    biggest = max(((len(r), len(c)) for s in S for r, c, _ in s.plan.sectors), key=lambda x: x[0] * x[1])
    in_blocks = sum(len(r) * len(c) for s in S for r, c, _ in s.plan.sectors) / sum(np.prod(s.shape) for s in S)
    flop_ratio = sum(map(dense_flops, S)) / sum(map(block_flops, S))
    print(f"{L:2d}  {str(chi):>5}  {len(S):6d}  {str(largest):>9}  {np.median([np.prod(s.shape) for s in S]):14.0f}"
          f"  {np.mean(blocks):12.1f} ({max(blocks)})  {str(biggest):>13}  {in_blocks:17.2f}  {flop_ratio:10.1f}")
    zero_outside += [outside_blocks(s) for s in S]
max(zero_outside) == 0.0

The matrices are small. At the production bond dimensions $M$ is at most $2\chi_w\times2\chi_w$ and splits into two to five blocks, so a truncated split works on blocks of a few rows. Only the exact conversion and $\chi_w\ge32$ produce blocks large enough for flops to matter.

## Five ways to split

On identical inputs:

* **(a) `m.split_pair`**, whatever the imported module currently does. It is only ever called, never re-implemented, so this column always measures production.
* **(b) blocked QR/eigh**: a fixed copy of the QR-based `m.split_pair` of 2026-09-24. Exact splits: one QR per block. Truncated splits: QR, then $R_qR_q^T = V\Lambda V^T$ and keep the top $k_q$ eigenvectors, $A_q = Q_qV_k$, $B_q = V_k^TR_q$ (arXiv:2212.09782). The columns of $Q_qV$ are the left singular vectors of $M_q$ because $M_qM_q^T = Q_q(R_qR_q^T)Q_q^T$, and $\Lambda = s^2$.
* **(c) blocked SVD**: the same blocks and kept ranks, one SVD per block, $A_q = U_{q,k}$, $B_q = S_{q,k}V_{q,k}^T$.
* **(d) dense SVD**, the naive split: one SVD of the whole $M$, keeping the same total number $k=\sum_q k_q$ of vectors (the global top $k$), no charge bookkeeping.
* **(e) dense QR/eigh**: (b)'s algorithm on the whole $M$. It is only a speed reference: it squares the singular values of the whole matrix, so directions with $s < 10^{-8}s_{\max}$ are not resolved, and it is left out of the correctness checks.

(d)/(a) is the question asked. (d)/(c) isolates blocking with SVD, (e)/(b) blocking with QR/eigh, (c)/(b) the per-block algorithm, and (a)/(b) any change made to `m.split_pair` since this notebook was written.

(b) to (e) take an array-module argument, so the same code runs in NumPy for the single-matrix comparison. The check confirms that (b) and the current `m.split_pair` produce the same split, on every seventh split of a truncated and of the exact conversion, in JAX and in NumPy. It compares the gauge-invariant $AB$ and projector $AA^T$, since eigenvectors are fixed only up to sign and different algorithms or roundings may flip them.

In [ ]:
def blocked_qr_split(T, ql, qr, kept=None, xp=jnp):
    # (b): QR per charge block, plus eigh(R R^T) when truncating; returns A (2Dl x k), B (k x 2Dr)
    Dl, _, _, Dr = T.shape
    M = T.reshape(2 * Dl, 2 * Dr)
    plan = m.sector_plan(ql, qr, kept)
    left, right = [], []
    for rows, columns, rank in plan.sectors:
        q, r = xp.linalg.qr(M[np.ix_(rows, columns)], mode="reduced")
        if kept is None:
            left.append(q[:, :rank])
            right.append(r[:rank])
        else:
            _, vectors = xp.linalg.eigh(r @ r.T)
            basis = vectors[:, ::-1][:, :rank]
            left.append(q @ basis)
            right.append(basis.T @ r)
    return m._assemble(left, right, plan, xp)


def blocked_svd_split(T, ql, qr, kept=None, xp=jnp):
    # (c): the same blocks and kept ranks, one SVD per block
    Dl, _, _, Dr = T.shape
    M = T.reshape(2 * Dl, 2 * Dr)
    plan = m.sector_plan(ql, qr, kept)
    left, right = [], []
    for rows, columns, rank in plan.sectors:
        u, s, vh = xp.linalg.svd(M[np.ix_(rows, columns)], full_matrices=False)
        left.append(u[:, :rank])
        right.append(s[:rank, None] * vh[:rank])
    return m._assemble(left, right, plan, xp)


def dense_svd_split(T, k, xp=jnp):
    # (d): one SVD of the whole matrix, keep the k largest singular values
    Dl, _, _, Dr = T.shape
    u, s, vh = xp.linalg.svd(T.reshape(2 * Dl, 2 * Dr), full_matrices=False)
    return u[:, :k], s[:k, None] * vh[:k]


def dense_qr_eigh_split(T, k, xp=jnp):
    # (e): one QR of the whole matrix, then the top k eigenvectors of R R^T
    Dl, _, _, Dr = T.shape
    q, r = xp.linalg.qr(T.reshape(2 * Dl, 2 * Dr), mode="reduced")
    _, vectors = xp.linalg.eigh(r @ r.T)
    basis = vectors[:, ::-1][:, :k]
    return q @ basis, basis.T @ r


def production_split(s, T):
    # (a): m.split_pair on one tensor, returned as matrices A (2Dl x k) and B (k x 2Dr)
    A, B, _ = m.split_pair(jnp.asarray(T), s.ql, s.qr, s.kept)
    return np.asarray(A).reshape(-1, s.k), np.asarray(B).reshape(s.k, -1)


numpy_splits = {"blocked QR/eigh": lambda s, T: blocked_qr_split(T, s.ql, s.qr, s.kept, xp=np),
                "blocked SVD":     lambda s, T: blocked_svd_split(T, s.ql, s.qr, s.kept, xp=np),
                "dense SVD":       lambda s, T: dense_svd_split(T, s.k, xp=np),
                "dense QR/eigh":   lambda s, T: dense_qr_eigh_split(T, s.k, xp=np)}

checks = []
for key in [(L, chi) for L, chi in GRID if chi is not None][-1:] + [(L, chi) for L, chi in GRID if chi is None]:
    for s in configs[key]["splits"][::7]:
        A, B = production_split(s, s.T[0])
        for xp in (jnp, np):
            A2, B2 = blocked_qr_split(xp.asarray(s.T[0]), s.ql, s.qr, s.kept, xp=xp)
            checks.append(np.allclose(A @ B, A2 @ B2, atol=1e-10) and np.allclose(A @ A.T, A2 @ A2.T, atol=1e-10))
len(checks), all(checks)

## Correctness

**Exact splits reconstruct $M$.** On every split of the exact L=20 conversion, for the first 10 walkers: the relative error $\|AB-M\|/\|M\|$ and $\|A^TA-1\|$ for `m.split_pair` and for (b) to (d). The dense SVD keeps $k=\sum_q\min(m_q,n_q)$ vectors, the rank the charges allow, which is less than $\min(m,n)$.

In [ ]:
exact_key = [(L, chi) for L, chi in GRID if chi is None][0]
exact_methods = {"m.split_pair": production_split,
                 **{name: numpy_splits[name] for name in ("blocked QR/eigh", "blocked SVD", "dense SVD")}}
worst = {name: [0.0, 0.0] for name in exact_methods}
for s in configs[exact_key]["splits"]:
    for T in s.T[:10]:
        M = T.reshape(s.shape)
        for name, split in exact_methods.items():
            A, B = split(s, T)
            worst[name][0] = max(worst[name][0], np.linalg.norm(A @ B - M) / np.linalg.norm(M))
            worst[name][1] = max(worst[name][1], np.abs(A.T @ A - np.eye(s.k)).max())
for name, (err, iso) in worst.items():
    print(f"{name:16s} max |AB - M|/|M| = {err:.1e}   max |A^T A - 1| = {iso:.1e}")
all(err < 1e-12 and iso < 1e-12 for err, iso in worst.values())

**Truncation on the reference determinant.** `m.plan_bonds` freezes the per-block counts by taking the global top $\chi_w$ Schmidt values of the *reference* determinant, in the same gauge `m.channel_mps` uses. So when the reference itself is converted, the blocked counts coincide with the global top-$k$ choice, and `m.split_pair` and the dense SVD must keep the same singular values and give the same truncated matrix $AB$, the best rank-$k$ approximation. $A$ is an isometry, so the singular values the blocked split keeps are those of $B$.

Two kinds of split are left out of the comparison, and counted. *Ties*: where $s_k - s_{k+1} < 10^{-8}s_1$ the best rank-$k$ approximation is not unique, and which of the tied vectors a method keeps is decided by rounding. *Off-plan*: `plan_bonds` (a NumPy SVD dry run) and `channel_mps` (JAX, `m.split_pair`) can settle a tie differently, and from then on the conversion follows a slightly different, equally good path, on which the frozen counts need not be the top $k$ of the matrix actually seen. There the blocked split keeps what the plan says and the dense SVD keeps the matrix's own top $k$, so their kept singular values differ by the small gap at the cut. `own_top_k_counts` finds these splits. In the development run, at $\chi_w = 64$ the cut lay near $s_k\sim10^{-7}$, where a truncation through $RR^T$ or $MM^T$, which squares the singular values, pins down the kept subspace only to about $10^{-9}$; elsewhere the agreement was at machine precision.

In [ ]:
def own_top_k_counts(s, T):
    # how the matrix's own global top-k singular values fall into the blocks
    values = [np.linalg.svd(T.reshape(s.shape)[np.ix_(r, c)], compute_uv=False)
              for r, c, _ in m.sector_plan(s.ql, s.qr).sectors]
    owner = np.concatenate([np.full(len(v), i) for i, v in enumerate(values)])
    top = owner[np.argsort(-np.concatenate(values), kind="stable")[:s.k]]
    return tuple(int((top == i).sum()) for i in range(len(values)))


def compare_on_reference(key, gap=1e-8):
    cfg = configs[key]
    worst_s, worst_M, ties, off_plan, compared = 0.0, 0.0, 0, 0, 0
    for T, ql, qr, kept in record_splits(cfg["C_ref"], cfg["plan"], cfg["bond_plan"]):
        s = Split(np.asarray(T), ql, qr, kept)
        all_s = np.linalg.svd(s.T.reshape(s.shape), compute_uv=False)
        if s.k < len(all_s) and all_s[s.k - 1] - all_s[s.k] < gap * all_s[0]:
            ties += 1
            continue
        if own_top_k_counts(s, s.T) != tuple(kept):
            off_plan += 1
            continue
        A, B = production_split(s, s.T)
        U, Sd = dense_svd_split(s.T, s.k, xp=np)
        worst_s = max(worst_s, np.abs(np.sort(np.linalg.svd(B, compute_uv=False)) - all_s[:s.k][::-1]).max())
        worst_M = max(worst_M, np.abs(A @ B - U @ Sd).max())
        compared += 1
    return worst_s, worst_M, compared, ties, off_plan


results = {key: compare_on_reference(key) for key in configs if key[1] is not None}
for (L, chi), (ds, dM, n, ties, off) in results.items():
    print(f"L={L} chi_w={chi:3d}: {n:3d} splits compared, max singular value difference {ds:.1e}, "
          f"max |AB_blocked - AB_dense| {dM:.1e}   (left out: {ties} ties, {off} off-plan)")
all(ds < 1e-8 and dM < 1e-8 for ds, dM, *_ in results.values())

**The dense SVD is a speed reference, not a drop-in.** It returns no bond labels, and they cannot always be recovered afterwards. A singular vector of a non-degenerate singular value lies inside one charge block, so its charge can be read off its support. But when two blocks share a singular value (common: particle–hole and reflection symmetry of the chain), LAPACK returns arbitrary mixtures of the two blocks' vectors, which carry no definite charge. The next centre move, the charge-blocked contraction with the trial and the static shapes under `jit` all need those labels. Below, *leaky* counts dense left singular vectors with more than $10^{-6}$ of their weight outside their dominant charge.

On walkers there is a second difference. The frozen counts `kept` were chosen on the reference; a walker's own global top-$k$ often distributes the $k$ vectors differently over the blocks. A dense truncation would follow the walker, so the bond's block sizes would vary from walker to walker and could not have static shapes. *top-k ≠ frozen* counts the (split, walker) pairs where that happens, over 20 walkers.

In [ ]:
def charge_leak(s, U):
    # 1 - largest weight of each left singular vector inside a single row charge
    row_charge = (np.asarray(s.ql)[:, None] + np.arange(2)).ravel()
    weights = np.stack([(U[row_charge == q] ** 2).sum(0) for q in np.unique(row_charge)])
    return 1.0 - weights.max(0)


print(" L  chi_w   leaky vectors (reference)   leaky vectors (walkers)   top-k != frozen (walkers)")
for key, cfg in configs.items():
    if key[1] is None:
        continue
    ref = [Split(np.asarray(T), ql, qr, kept) for T, ql, qr, kept in
           record_splits(cfg["C_ref"], cfg["plan"], cfg["bond_plan"])]
    leak_ref = np.concatenate([charge_leak(s, dense_svd_split(s.T, s.k, xp=np)[0]) for s in ref])
    leak_w = np.concatenate([charge_leak(s, dense_svd_split(T, s.k, xp=np)[0])
                             for s in cfg["splits"] for T in s.T[:20]])
    moved = [own_top_k_counts(s, T) != tuple(s.kept) for s in cfg["splits"] for T in s.T[:20]]
    print(f"{key[0]:2d}  {key[1]:5d}   {np.mean(leak_ref > 1e-6):8.1%} of {len(leak_ref):5d}"
          f"            {np.mean(leak_w > 1e-6):8.1%} of {len(leak_w):6d}       {np.mean(moved):8.1%} of {len(moved)}")

## Timing in the production setting

In `mps_cpmc_new.py` the conversion runs inside `jax.jit(jax.vmap(...))` over the walker population. On CPU, `vmap` turns each LAPACK call into one batched custom call that loops over the walkers, so the number of *calls* per split is set by the block structure, not by the batch size.

Each split is compiled as `jax.jit(jax.vmap(split))` for its recorded batch of 200 walkers and timed: one warm-up call, then the median of at least 5 repeats (more for fast calls, up to 0.2 s of repeats), each ended by `jax.block_until_ready`. The number of LAPACK kernels per walker is read off the compiled program (its `lapack_*` custom calls), so it is counted, not assumed, for `m.split_pair` too. Splits with the same block structure (same $T$ shape and the same rows, columns and rank per block) run the same kernels on the same shapes; only the gather indices differ. So one representative per structure is timed and weighted by the number of splits that share it, which cuts about 2000 splits down to about 220 structures to compile. Summing over a conversion gives the split time per conversion of 200 walkers.

*floor* is the same measurement for a trivial jitted function on the same batch (`T + 1`): dispatch plus one pass over the data. It is paid by every method, and in production, where the whole conversion is one XLA program, most of it disappears.

In [ ]:
def median_time(f, *args, min_repeats=5, max_repeats=50, budget=0.2):
    jax.block_until_ready(f(*args))                      # warm up
    times = []
    while len(times) < max_repeats and (len(times) < min_repeats or sum(times) < budget):
        t0 = time.perf_counter()
        jax.block_until_ready(f(*args))
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


def structure(s):
    return s.T.shape[1:], tuple((len(r), len(c), rank) for r, c, rank in s.plan.sectors)


def representatives(splits):
    # first split of each block structure, and how many splits share that structure
    groups = {}
    for s in splits:
        groups.setdefault(structure(s), [s, 0])[1] += 1
    return list(groups.values())


NAMES = ["m.split_pair", "blocked QR/eigh", "blocked SVD", "dense SVD", "dense QR/eigh", "floor"]


def jax_split_times(s):
    # seconds per jit(vmap) call on the batch s.T, and LAPACK kernels per walker, for every method
    fns = {"m.split_pair":    lambda T: m.split_pair(T, s.ql, s.qr, s.kept)[:2],
           "blocked QR/eigh": lambda T: blocked_qr_split(T, s.ql, s.qr, s.kept),
           "blocked SVD":     lambda T: blocked_svd_split(T, s.ql, s.qr, s.kept),
           "dense SVD":       lambda T: dense_svd_split(T, s.k),
           "dense QR/eigh":   lambda T: dense_qr_eigh_split(T, s.k),
           "floor":           lambda T: T + 1.0}
    T = jnp.asarray(s.T)
    times, calls = {}, {}
    for name in NAMES:
        compiled = jax.jit(jax.vmap(fns[name])).lower(T).compile()
        calls[name] = compiled.as_text().count('custom_call_target="lapack_')
        times[name] = median_time(compiled, T)
    return times, calls


jax_times = {}                                   # (L, chi) -> list of (split, count, times, calls)
for key, cfg in configs.items():
    t0 = time.perf_counter()
    jax_times[key] = [(s, count, *jax_split_times(s)) for s, count in representatives(cfg["splits"])]
    print(f"L={key[0]} chi_w={key[1]}: {len(jax_times[key])} structures timed in {time.perf_counter() - t0:.0f} s")

**Split time per conversion** (ms for 200 walkers, sum over all splits), the LAPACK kernels per walker per conversion counted from the compiled programs, and ratios: above 1 the method in the denominator is faster. (d)/(a) is the question asked (naive dense SVD against production). (d)/(c) is blocking alone with SVD, (e)/(b) blocking alone with QR/eigh, and (a)/(b) compares the current `m.split_pair` with the QR/eigh version.

In [ ]:
def per_conversion(timed):
    total = {name: sum(count * t[name] for _, count, t, _ in timed) for name in NAMES}
    calls = {name: sum(count * c[name] for _, count, _, c in timed) for name in NAMES}
    return total, calls


print(" L  chi_w |   (a) m.     (b) blk    (c) blk    (d) dense  (e) dense   floor |    LAPACK kernels per walker   |"
      "  (d)/(a) (d)/(c) (e)/(b) (a)/(b)")
print("        | split_pair   QR/eigh      SVD        SVD      QR/eigh    (ms) |    a /    b /    c /   d /    e  |")
summary = {}
for (L, chi), timed in jax_times.items():
    total, calls = per_conversion(timed)
    summary[L, chi] = total
    ms = [1e3 * total[n] for n in NAMES]
    c = [calls[n] for n in NAMES[:5]]
    r = lambda a, b: total[a] / total[b]
    print(f"{L:2d}  {str(chi):>5} |" + "".join(f"{x:10.1f} " for x in ms[:5]) + f"{ms[5]:7.1f} |"
          f" {c[0]:5d} /{c[1]:5d} /{c[2]:5d} /{c[3]:4d} /{c[4]:5d} |"
          f" {r('dense SVD', 'm.split_pair'):7.2f} {r('dense SVD', 'blocked SVD'):7.2f}"
          f" {r('dense QR/eigh', 'blocked QR/eigh'):7.2f} {r('m.split_pair', 'blocked QR/eigh'):7.2f}")

## Where the time goes, split by split

Each point is one block structure: its time per walker (the batched call divided by 200) against the number of entries of $M$, for all configurations. The right panel is the ratio dense SVD / `m.split_pair` for the same splits; above the line the blocked split is faster.

In [ ]:
colors = {"m.split_pair": "#2a78d6", "blocked SVD": "#eb6834", "dense SVD": "#1baf7a"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, color in colors.items():
    x = [np.prod(s.shape) for timed in jax_times.values() for s, _, _, _ in timed]
    y = [1e6 * t[name] / N_BATCH for timed in jax_times.values() for _, _, t, _ in timed]
    ax1.scatter(x, y, s=18, color=color, alpha=0.75, label=name, edgecolors="none")
for truncated, marker, label in ((True, "o", "truncated splits"), (False, "s", "exact splits")):
    pts = [(np.prod(s.shape), t["dense SVD"] / t["m.split_pair"])
           for timed in jax_times.values() for s, _, t, _ in timed if (s.kept is not None) == truncated]
    if pts:
        ax2.scatter(*zip(*pts), s=22, marker=marker, color="#2a78d6", alpha=0.75, label=label, edgecolors="none")
ax2.axhline(1.0, color="0.4", lw=1)
for ax in (ax1, ax2):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("entries of M = (2 Dl)(2 Dr)")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
ax1.set_ylabel(f"time per walker per split (µs), batch of {N_BATCH}")
ax2.set_ylabel("dense SVD time / m.split_pair time")
ax1.set_title("per-split time")
ax2.set_title("blocked wins above 1")
fig.tight_layout()
plt.show()

**Flops vs measured.** If time followed flops, dense / blocked with the same algorithm, (d)/(c) and (e)/(b), would equal the flop ratio from the matrix table. Where the measured ratio falls short of it, per-call overhead rather than flops sets the time.

In [ ]:
print(" L  chi_w   flop ratio   measured (d)/(c)   measured (e)/(b)")
for key, cfg in configs.items():
    S = cfg["splits"]
    flop_ratio = sum(map(dense_flops, S)) / sum(map(block_flops, S))
    total = summary[key]
    print(f"{key[0]:2d}  {str(key[1]):>5}  {flop_ratio:11.1f}  {total['dense SVD'] / total['blocked SVD']:17.2f}"
          f"  {total['dense QR/eigh'] / total['blocked QR/eigh']:17.2f}")

## Second view: NumPy, one matrix at a time

(b) to (e) in NumPy on a single matrix (walker 0), each timed as the median of 5 samples of enough back-to-back calls to fill 10 ms. `m.split_pair` is JAX-only, so it has no column here; (b) is the blocked split. Every LAPACK call now also pays Python overhead of a few µs, so per-call cost weighs much more relative to flops than under `jit(vmap)`. Both views call the same LAPACK, Accelerate (JAX reaches it through SciPy's `cython_lapack`), so they differ in calling overhead, not in the library.

In [ ]:
def numpy_time(f, *args, target=0.01, repeats=5):
    f(*args)
    number = 1
    while True:
        t0 = time.perf_counter()
        for _ in range(number):
            f(*args)
        if time.perf_counter() - t0 > target:
            break
        number *= 4
    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        for _ in range(number):
            f(*args)
        samples.append((time.perf_counter() - t0) / number)
    return float(np.median(samples))


print(" L  chi_w |  (b) blocked   (c) blocked   (d) dense   (e) dense  (ms per conversion, one walker) |  (d)/(b)  (e)/(b)")
numpy_summary = {}
for key, cfg in configs.items():
    total = dict.fromkeys(numpy_splits, 0.0)
    for s, count in representatives(cfg["splits"]):
        for name, split in numpy_splits.items():
            total[name] += count * numpy_time(split, s, s.T[0])
    numpy_summary[key] = total
    ms = {k: 1e3 * v for k, v in total.items()}
    print(f"{key[0]:2d}  {str(key[1]):>5} | {ms['blocked QR/eigh']:11.2f}  {ms['blocked SVD']:11.2f}  {ms['dense SVD']:11.2f}"
          f"  {ms['dense QR/eigh']:10.2f}                                  | {total['dense SVD'] / total['blocked QR/eigh']:7.2f}"
          f"  {total['dense QR/eigh'] / total['blocked QR/eigh']:7.2f}")

## Conclusions

*These conclusions come from a development run, not from the outputs above: single-threaded (`OMP/VECLIB=1`, XLA intra-op 1), `nice -n 15`, on a shared machine with other benchmarks running, when `m.split_pair` was still the QR/eigh version, so (a) = (b). Absolute times are unreliable; the ratios are indicative. Check them against the outputs above.*

Per conversion under `jit(vmap)`, batch 200, ms (development run):

| L | $\chi_w$ | (a)=(b) blocked QR/eigh | (c) blocked SVD | (d) dense SVD | (e) dense QR/eigh | (d)/(a) | (d)/(c) | (e)/(b) | flop ratio |
|---|---|---|---|---|---|---|---|---|---|
| 20 | exact | 90 | 501 | 2254 | 1606 | 24.9 | 4.5 | 17.8 | 19.3 |
| 32 | 4 | 93 | 245 | 271 | 159 | 2.9 | 1.1 | 1.7 | 9.2 |
| 32 | 8 | 173 | 397 | 610 | 364 | 3.5 | 1.5 | 2.1 | 14.2 |
| 32 | 16 | 342 | 573 | 1482 | 975 | 4.3 | 2.6 | 2.9 | 17.2 |
| 32 | 32 | 626 | 1008 | 3769 | 3094 | 6.0 | 3.7 | 4.9 | 20.0 |
| 32 | 64 | 1049 | 1786 | 8785 | 6960 | 8.4 | 4.9 | 6.6 | 21.9 |
| 48 | 4 | 150 | 429 | 445 | 269 | 3.0 | 1.0 | 1.8 | 10.4 |
| 48 | 8 | 308 | 722 | 1206 | 722 | 3.9 | 1.7 | 2.4 | 16.3 |
| 48 | 16 | 643 | 1087 | 3119 | 1908 | 4.9 | 2.9 | 3.0 | 18.0 |

* **In the production setting the blocked split won everywhere, with no crossover in $\chi_w$.** Dense SVD / blocked QR/eigh was 2.9–3.0 at $\chi_w=4$ (8×8 matrices), 3.5–3.9 at 8, 4.3–4.9 at 16, 6–8 at 32–64, and 25 for the exact conversion. The advantage grew with matrix size.
* **Two effects multiply.** *Algorithm*: at equal blocking, QR/eigh beat SVD by 1.6–2.9× on truncated splits and 5.6× on exact ones, where only a QR is needed ((c)/(b)). JAX's batched `gesdd` is expensive on small matrices. *Blocking*: at equal algorithm, blocking gained only 1.0–1.1× at $\chi_w=4$ with SVD ((d)/(c)), although it saves 9–10× in flops. About three SVD calls on blocks of at most 3×3, instead of one on the 8×8 matrix, pay the fixed cost per call three times, which cancels the flop saving. The gain rose to 2.6–2.9× at $\chi_w=16$ and about 5× at 64, still well short of the 17–22× flop ratio.
* **One LAPACK call at a time from Python, the verdict reverses** (NumPy view, development run): dense SVD was 3.4–9× *faster* than blocked QR/eigh for $\chi_w\le16$, 1.9× faster at 32, and roughly even (ratio 0.8–0.9) at $\chi_w=64$ and for the exact conversion. Each call pays microseconds of Python overhead, and the blocked split makes 8–12× more LAPACK calls. This is the regime where "dense beats blocked at small sizes" (as seen for the overlap contraction) holds. `jit(vmap)` amortises the call overhead over the batch, so it does not apply to production.
* **Dense is not a drop-in replacement anyway.** It loses the charge labels: 0.3–1.6% of the dense singular vectors on the reference mixed charges (degenerate values across blocks). On 13–35% of (split, walker) pairs, the walker's own top $k$ fell into the blocks differently from the frozen counts, so dense truncation would break static shapes.